# MySQL InnoDB Storage Engine, Clustered Indexes & Binary Logs

Interactive hands-on sandbox exploring internal storage mechanics, algorithmic data structures, and architectural invariants.


In [ ]:
import sys
from pathlib import Path

# Prepend project_solution to sys.path so Track A internal engines import cleanly
ps_dir = Path('.').resolve() / 'project_solution'
if str(ps_dir) not in sys.path:
    sys.path.insert(0, str(ps_dir))

print(f'Python runtime: {sys.version.split()[0]}')
print('Loaded internal mechanics for: Module_06_MySQL_MariaDB_InnoDB_Replication')


## 1. Engine Initialization & Setup

Importing the module's Track A internal simulation engine and instantiating state.


In [ ]:
from innodb_engine import InnoDBClusteredTable

# Initialize InnoDB Clustered Table with Primary Key 'id'
table = InnoDBClusteredTable(table_name="users", primary_key="id")
table.create_secondary_index("email")

# Insert records
table.insert({"id": 10, "email": "alice@corp.com", "role": "admin"})
table.insert({"id": 5, "email": "bob@corp.com", "role": "developer"})
table.insert({"id": 20, "email": "charlie@corp.com", "role": "dba"})

print(f"Clustered B+ Tree keys (Strictly sorted by PK): {list(table._clustered_index.keys())}")


## 2. Core Architectural Operations & State Mutation

Executing data mutations, transactions, or indexing procedures.


In [ ]:
# Secondary Index Bookmark Lookup:
# Secondary index maps 'email' -> 'id' (PK bookmark).
# Querying non-indexed columns ('role') performs bookmark lookup into the clustered tree.
row, was_covering = table.get_by_secondary_index("email", "bob@corp.com")
print(f"Secondary index lookup result: {row}, was_covering: {was_covering}")


## 3. Performance Micro-Benchmarking & Invariant Verification

Evaluating execution latency, cache hits, or computational trade-offs.


In [ ]:
import time

# Benchmark clustered primary key seek vs secondary index bookmark lookup
t0 = time.perf_counter()
for _ in range(5000):
    _ = table.get_by_primary_key(10)
t_pk = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
for _ in range(5000):
    _ = table.get_by_secondary_index("email", "bob@corp.com")
t_sec = (time.perf_counter() - t0) * 1000

print(f"5,000 Clustered PK Lookups: {t_pk:.2f} ms")
print(f"5,000 Secondary Bookmark Lookups: {t_sec:.2f} ms (Two-step index traversal)")


## 4. Architectural Invariant Verification

Asserting mathematical correctness and durability invariants.


In [ ]:
# Verify invariants
assert sorted(table._clustered_index.keys()) == [5, 10, 20], "Clustered index must contain all primary keys"
assert row["id"] == 5 and row["role"] == "developer"
assert was_covering is False, "Querying all columns via secondary index requires bookmark lookup"
print("[+] InnoDB Clustered Index & Secondary Bookmark invariants verified successfully!")


## Summary & Operational DBRE Best Practices

1. **Never bypass serialization contracts:** Always enforce binary-safe schemas and validated boundaries.
2. **Monitor buffer and memory allocations:** Understand the latency cliff when in-memory structures spill to disk.
3. **Ensure idempotency across replication tiers:** Distributed operations must survive retries without corrupting state.
